In [ ]:
"""
Jupyter-friendly version for moving low-quality recordings
Copy and run each cell as needed
"""

import pandas as pd
from pathlib import Path
from datetime import datetime
import shutil
import json


CSV_PATH = "/data/big_rim/uploade_ssh_mir_dataset/oct3v1_metadata/export_mapping_master.csv"
LOG_DIR = Path("/home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs")
DEST_BASE = Path("/data/big_rim/uploade_ssh_mir_dataset/oct3v1_beh_only")
LOW_QUALITY_DIR = Path("/data/big_rim/uesless/oct3v1_exported_abandon/low_quality")

# Source rec paths to move
LOW_QUALITY_SOURCES = [
    "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03/20240819V1r2",
    "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_04/20240819V1r1_20_10",
    "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_07/20240916v1r1_15_05_30min",
    "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_13/240605pmc_righthole_cricket_acrylic_test_15_05",
    "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_30/2social_mini_0605pmc_single_15_38",
]


def find_dest_paths(csv_path, source_fragments):
    """Find destination paths for given source path fragments."""
    df = pd.read_csv(csv_path)
    
    matches = []
    for fragment in source_fragments:
        # Find rows where source_rec_path contains the fragment
        mask = df['source_rec_path'].str.contains(fragment, na=False)
        matching_rows = df[mask]
        
        if len(matching_rows) == 0:
            print(f"⚠️  WARNING: No match found for '{fragment}'")
            matches.append({
                'fragment': fragment,
                'source_rec_path': None,
                'dest_rel_path': None,
                'found': False
            })
        else:
            for _, row in matching_rows.iterrows():
                matches.append({
                    'fragment': fragment,
                    'source_rec_path': row['source_rec_path'],
                    'dest_rel_path': row['dest_rel_path'],
                    'animal_id': row.get('animal_id', ''),
                    'is_social': row.get('is_social', ''),
                    'found': True
                })
    
    return matches



def display_matches(matches):
    """Display the matched paths nicely."""
    print("\n" + "="*80)
    print("FOLDERS TO BE MOVED TO LOW_QUALITY")
    print("="*80)
    
    found_count = 0
    for i, match in enumerate(matches, 1):
        if match['found']:
            found_count += 1
            print(f"\n{i}. Fragment: {match['fragment']}")
            print(f"   Source:      {match['source_rec_path']}")
            print(f"   Destination: {match['dest_rel_path']}")
            print(f"   Animal:      {match['animal_id']} (social={match['is_social']})")
        else:
            print(f"\n{i}. Fragment: {match['fragment']}")
            print(f"   ❌ NOT FOUND IN CSV")
    
    print("\n" + "="*80)
    print(f"Total fragments: {len(matches)}")
    print(f"Found in CSV: {found_count}")
    print(f"Not found: {len(matches) - found_count}")
    print("="*80 + "\n")
    
    return matches



def move_folders(matches, dry_run=True):
    """
    Move the folders and create logs.
    
    Parameters:
    -----------
    matches : list
        List of match dictionaries from find_dest_paths()
    dry_run : bool
        If True, just show what would happen without moving anything
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = LOG_DIR / f"move_low_quality_{timestamp}.json"
    
    # Create directories if they don't exist
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    if not dry_run:
        LOW_QUALITY_DIR.mkdir(parents=True, exist_ok=True)
    
    results = []
    
    print("\n" + "="*80)
    if dry_run:
        print("DRY RUN MODE - No actual moves will be performed")
    else:
        print("MOVING FOLDERS...")
    print("="*80 + "\n")
    
    for match in matches:
        if not match['found']:
            results.append({
                **match,
                'status': 'skipped',
                'reason': 'not found in CSV'
            })
            continue
        
        # Construct full paths
        src_full = DEST_BASE / match['dest_rel_path']
        dest_full = LOW_QUALITY_DIR / match['dest_rel_path']
        
        result = {
            **match,
            'src_full_path': str(src_full),
            'dest_full_path': str(dest_full),
            'timestamp': timestamp,
        }
        
        # Check if source exists
        if not src_full.exists():
            result['status'] = 'error'
            result['reason'] = f'Source does not exist'
            print(f"❌ {match['dest_rel_path']}: source not found at {src_full}")
            results.append(result)
            continue
        
        # Check if destination already exists
        if dest_full.exists():
            result['status'] = 'error'
            result['reason'] = f'Destination already exists'
            print(f"❌ {match['dest_rel_path']}: destination already exists")
            results.append(result)
            continue
        
        # Move the folder
        try:
            if dry_run:
                print(f"[DRY RUN] Would move:")
                print(f"  FROM: {src_full}")
                print(f"  TO:   {dest_full}")
                result['status'] = 'dry_run'
            else:
                dest_full.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(src_full), str(dest_full))
                print(f"✓ Moved: {match['dest_rel_path']}")
                result['status'] = 'success'
        except Exception as e:
            result['status'] = 'error'
            result['reason'] = str(e)
            print(f"❌ Error moving {match['dest_rel_path']}: {e}")
        
        results.append(result)
    
    # Save log
    log_data = {
        'execution_time': timestamp,
        'dry_run': dry_run,
        'total_folders': len(matches),
        'moved': sum(1 for r in results if r['status'] == 'success'),
        'errors': sum(1 for r in results if r['status'] == 'error'),
        'skipped': sum(1 for r in results if r['status'] == 'skipped'),
        'dry_runs': sum(1 for r in results if r['status'] == 'dry_run'),
        'results': results
    }
    
    with open(log_file, 'w') as f:
        json.dump(log_data, f, indent=2)
    
    # Summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    if dry_run:
        print("This was a DRY RUN - no files were actually moved")
        print(f"Would move: {log_data['dry_runs']} folders")
    else:
        print(f"Successfully moved: {log_data['moved']} folders")
    print(f"Errors: {log_data['errors']}")
    print(f"Skipped: {log_data['skipped']}")
    print(f"\n📝 Log saved to: {log_file}")
    print("="*80)
    
    return results



In [2]:
# Run this to see what would be moved
matches = find_dest_paths(CSV_PATH, LOW_QUALITY_SOURCES)
display_matches(matches)


FOLDERS TO BE MOVED TO LOW_QUALITY

1. Fragment: /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03/20240819V1r2
   Source:      /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03/20240819V1r2
   Destination: single/20241003_VC19_S2/
   Animal:      VC19 (social=0)

2. Fragment: /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_04/20240819V1r1_20_10
   Source:      /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_04/20240819V1r1_20_10
   Destination: single/20241004_VC4_S4/
   Animal:      VC4 (social=0)

3. Fragment: /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_07/20240916v1r1_15_05_30min
   Source:      /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_07/20240916v1r1_15_05_30min
   Destination: single/20241007_VC10_S2/
   Animal:      VC10 (social=0)

4. Fragment: /data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_13/240605pmc_righthole_cricket_acrylic_test_15_05
   Source:      /data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_13/240605pmc_righthole_cricket_acrylic_test_15_05
   Destination: single/20241113_MC10_S1/
   Animal:

[{'fragment': '/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03/20240819V1r2',
  'source_rec_path': '/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03/20240819V1r2',
  'dest_rel_path': 'single/20241003_VC19_S2/',
  'animal_id': 'VC19',
  'is_social': 0,
  'found': True},
 {'fragment': '/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_04/20240819V1r1_20_10',
  'source_rec_path': '/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_04/20240819V1r1_20_10',
  'dest_rel_path': 'single/20241004_VC4_S4/',
  'animal_id': 'VC4',
  'is_social': 0,
  'found': True},
 {'fragment': '/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_07/20240916v1r1_15_05_30min',
  'source_rec_path': '/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_07/20240916v1r1_15_05_30min',
  'dest_rel_path': 'single/20241007_VC10_S2/',
  'animal_id': 'VC10',
  'is_social': 0,
  'found': True},
 {'fragment': '/data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_13/240605pmc_righthole_cricket_acrylic_test_15_05',
  'source_rec_path': '/data/big_rim/rsync_dcc_sum/Oct3V1/2024

In [7]:
# Run this to do a dry run (safe - doesn't actually move anything)
results = move_folders(matches, dry_run=False)


MOVING FOLDERS...

✓ Moved: single/20241003_VC19_S2/
✓ Moved: single/20241004_VC4_S4/
✓ Moved: single/20241007_VC10_S2/
✓ Moved: single/20241113_MC10_S1/
✓ Moved: social/20241030_MC10+UNK1_S6/

SUMMARY
Successfully moved: 5 folders
Errors: 0
Skipped: 0

📝 Log saved to: /home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs/move_low_quality_20251107_130252.json


In [6]:
# Convert results to DataFrame for easier inspection
results_df = pd.DataFrame(results)
results_df[['fragment', 'dest_rel_path', 'status', 'animal_id']]

,fragment,dest_rel_path,status,animal_id
0,/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03/...,single/20241003_VC19_S2/,dry_run,VC19
1,/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_04/...,single/20241004_VC4_S4/,dry_run,VC4
2,/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_07/...,single/20241007_VC10_S2/,dry_run,VC10
3,/data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_13/...,single/20241113_MC10_S1/,dry_run,MC10
4,/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_30/...,social/20241030_MC10+UNK1_S6/,dry_run,MC10+UNK1
